# Lab 18 — UCI multicenter synthetic hospital-like augmentation

Mục tiêu: đánh giá việc bổ sung dữ liệu synthetic trên 920 bệnh nhân UCI bằng LOCO.

- Nền cố định: P1 sentinel-aware + unweighted model từ Lab 17.
- Synthetic chỉ được fit/sample trong training hospitals của từng vòng LOCO.
- Test hospital luôn là dữ liệu thật chưa từng dùng để sinh hoặc chọn dữ liệu.
- So sánh `real_only`, `clean_synthetic` và `hospital_like`. Tỷ lệ synthetic là synthetic / real-train.
- Đây là nghiên cứu robustness, không phải dữ liệu bệnh nhân độc lập hay xác nhận lâm sàng.

In [ ]:
!pip -q install sdv lightgbm scipy seaborn

import json
import time
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp
from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score,
    confusion_matrix, f1_score, precision_score, recall_score,
    roc_auc_score, brier_score_loss)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
FEATURES = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach',
            'exang','oldpeak','slope','ca','thal']
TARGET = 'target'
NUMERICAL = ['age','trestbps','chol','thalach','oldpeak']
CATEGORICAL = [c for c in FEATURES if c not in NUMERICAL]
SEEDS = [42, 52, 62]
AUGMENTATION_RATIOS = [0.25, 0.50, 1.00, 2.00]
OUTPUT_DIR = Path('/content/lab18_synthetic_augmentation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BASE_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease'
FILES = {'Cleveland':'processed.cleveland.data', 'Hungarian':'processed.hungarian.data',
         'Switzerland':'processed.switzerland.data', 'Long Beach VA':'processed.va.data'}
ALL_COLUMNS = FEATURES + [TARGET]

def read_uci(hospital, filename):
    frame = pd.read_csv(f'{BASE_URL}/{filename}', header=None, names=ALL_COLUMNS,
                        na_values=['?', ''], skipinitialspace=True)
    for column in FEATURES + [TARGET]:
        frame[column] = pd.to_numeric(frame[column], errors='coerce')
    frame = frame.dropna(subset=[TARGET]).copy()
    frame[TARGET] = (frame[TARGET] > 0).astype(int)
    frame['hospital'] = hospital
    return frame[FEATURES + [TARGET, 'hospital']]

data = pd.concat([read_uci(h, f) for h, f in FILES.items()], ignore_index=True)
assert len(data) == 920, f'Expected 920 rows, got {len(data)}'
SITES = list(FILES.keys())
print('Dataset:', data.shape)
display(data.groupby('hospital')[TARGET].agg(['size','sum','mean']).round(4))

## 1. P1 preprocessing và model evaluator

P1 chỉ xử lý sentinel `trestbps <= 0` và `chol <= 0` thành missing. Imputation, missing indicators và scaling được fit bên trong từng training fold. Threshold giữ ở 0.5 để so sánh công bằng với Lab 17.

In [ ]:
def apply_p1(frame):
    out = frame.copy()
    for column in FEATURES:
        out[column] = pd.to_numeric(out[column], errors='coerce')
    for column in ['trestbps', 'chol']:
        out.loc[out[column] <= 0, column] = np.nan
    return out

def safe_auc(y_true, probability):
    return float(roc_auc_score(y_true, probability)) if len(np.unique(y_true)) > 1 else np.nan

def build_model(model_name, seed):
    numeric = Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)),
                       ('scaler', StandardScaler())])
    categorical = Pipeline([('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
                            ('encoder', OneHotEncoder(handle_unknown='ignore'))])
    prep = ColumnTransformer([('numeric', numeric, NUMERICAL),
                             ('categorical', categorical, CATEGORICAL)])
    if model_name == 'Logistic Regression':
        estimator = LogisticRegression(max_iter=2000, random_state=42)
    else:
        estimator = LGBMClassifier(n_estimators=250, learning_rate=0.03, num_leaves=15,
            min_child_samples=15, subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            random_state=42, verbosity=-1)
    return Pipeline([('preprocess', prep), ('model', estimator)])

def fit_and_score(train_frame, test_frame, model_name, seed):
    train_p1, test_p1 = apply_p1(train_frame), apply_p1(test_frame)
    X_train, y_train = train_p1[FEATURES], train_p1[TARGET]
    X_test, y_test = test_p1[FEATURES], test_p1[TARGET]
    model = build_model(model_name, seed)
    started = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - started
    probability = model.predict_proba(X_test)[:, 1]
    prediction = (probability >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()
    return {'accuracy': accuracy_score(y_test, prediction),
        'precision': precision_score(y_test, prediction, zero_division=0),
        'recall': recall_score(y_test, prediction, zero_division=0),
        'pr_auc': average_precision_score(y_test, probability),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'f1': f1_score(y_test, prediction, zero_division=0),
        'roc_auc': safe_auc(y_test, probability),
        'brier': brier_score_loss(y_test, probability),
        'false_negatives': int(fn), 'fit_seconds': fit_seconds}


## 2. Gaussian synthetic và hospital-like corruption

Mỗi training fold fit một GaussianCopulaSynthesizer riêng cho từng lớp. `hospital_like` là stress profile đã định trước: missing 15% mỗi biến, missing bổ sung 35% ở `ca/thal`, outlier số 4%, rounding 20% và categorical code không hợp lệ 2%. Các lỗi này chỉ áp dụng cho synthetic rows.

In [ ]:
CLINICAL_BOUNDS = {'age': (18, 100), 'trestbps': (60, 260), 'chol': (80, 800),
                   'thalach': (40, 240), 'oldpeak': (0.0, 10.0)}
INTEGER_NUMERICAL = ['age', 'trestbps', 'chol', 'thalach']

def make_metadata(reference):
    metadata = Metadata.detect_from_dataframe(data=reference, table_name='patients')
    for column in CATEGORICAL:
        metadata.update_column(column_name=column, sdtype='categorical')
    for column in NUMERICAL:
        metadata.update_column(column_name=column, sdtype='numerical')
    metadata.validate()
    return metadata

def nearest_allowed(series, allowed):
    allowed = np.asarray(sorted(pd.Series(allowed).dropna().unique()), dtype=float)
    values = pd.to_numeric(series, errors='coerce').to_numpy(dtype=float)
    valid = ~np.isnan(values)
    if len(allowed) and valid.any():
        values[valid] = allowed[np.abs(values[valid, None] - allowed[None, :]).argmin(axis=1)]
    return values

def enforce_domains(sample, reference):
    result = sample.copy()
    for column, (low, high) in CLINICAL_BOUNDS.items():
        result[column] = pd.to_numeric(result[column], errors='coerce').clip(low, high)
    result[INTEGER_NUMERICAL] = result[INTEGER_NUMERICAL].round()
    result['oldpeak'] = result['oldpeak'].round(1)
    for column in CATEGORICAL:
        result[column] = nearest_allowed(result[column], reference[column])
    return result[FEATURES]

def fit_generators(train_frame, seed):
    np.random.seed(seed)
    generators = {}
    for label, class_data in train_frame.groupby(TARGET):
        reference = apply_p1(class_data[FEATURES]).reset_index(drop=True)
        for column in FEATURES:
            if reference[column].isna().any():
                fallback = reference[column].median()
                if pd.isna(fallback):
                    mode = reference[column].mode(dropna=True)
                    fallback = mode.iloc[0] if len(mode) else 0
                reference[column] = reference[column].fillna(fallback)
        model = GaussianCopulaSynthesizer(make_metadata(reference),
            enforce_min_max_values=True, enforce_rounding=True)
        model.fit(reference)
        generators[int(label)] = (model, reference)
    return generators

def corrupt_hospital_like(frame, seed):
    out = frame.copy()
    rng = np.random.default_rng(seed)
    for column in FEATURES:
        out.loc[rng.random(len(out)) < 0.15, column] = np.nan
    for column in ['ca', 'thal']:
        out.loc[rng.random(len(out)) < 0.35, column] = np.nan
    for column in NUMERICAL:
        mask = rng.random(len(out)) < 0.04
        factors = rng.choice([0.5, 2.0, 3.0], size=int(mask.sum()))
        out.loc[mask, column] = out.loc[mask, column].to_numpy() * factors
        round_mask = rng.random(len(out)) < 0.20
        out.loc[round_mask, column] = out.loc[round_mask, column].round(0 if column != 'oldpeak' else 1)
    for column in CATEGORICAL:
        out.loc[rng.random(len(out)) < 0.02, column] = 99
    return out

def sample_synthetic(generators, train_frame, n_rows, seed, hospital_like=False):
    rates = train_frame[TARGET].value_counts(normalize=True).reindex([0, 1], fill_value=0)
    n0 = int(round(n_rows * rates.loc[0]))
    counts = {0: n0, 1: n_rows - n0}
    parts = []
    for label, count in counts.items():
        if count == 0: continue
        model, reference = generators[label]
        sampled = enforce_domains(model.sample(num_rows=count), reference)
        sampled[TARGET] = label
        parts.append(sampled)
    result = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    if hospital_like:
        labels = result[TARGET].copy()
        result[FEATURES] = corrupt_hospital_like(result[FEATURES], seed + 10000)
        result[TARGET] = labels
    return result[FEATURES + [TARGET]]

## 3. Quality audit và LOCO experiment

KS/TV chỉ là audit chất lượng. Exact-match với test hospital được báo cáo riêng để phát hiện rò rỉ, tuyệt đối không dùng để chọn cấu hình.

In [ ]:
def total_variation(real, synthetic):
    categories = sorted(set(real.dropna().unique()) | set(synthetic.dropna().unique()))
    if not categories: return 0.0
    p = real.value_counts(normalize=True).reindex(categories, fill_value=0)
    q = synthetic.value_counts(normalize=True).reindex(categories, fill_value=0)
    return float(0.5 * np.abs(p - q).sum())

def exact_match_rate(candidate, reference):
    if len(candidate) == 0: return 0.0
    matches = candidate[ALL_COLUMNS].merge(reference[ALL_COLUMNS].drop_duplicates(), how='inner')
    return len(matches) / len(candidate)

def quality_audit(train_frame, synthetic_frame, test_frame):
    train_ref = apply_p1(train_frame[ALL_COLUMNS])
    synthetic_ref = apply_p1(synthetic_frame[ALL_COLUMNS])
    rows = []
    for column in NUMERICAL:
        left = train_ref[column].dropna(); right = synthetic_ref[column].dropna()
        distance = ks_2samp(left, right).statistic if len(left) and len(right) else np.nan
        rows.append({'feature': column, 'metric': 'numeric_KS', 'distance': distance})
    for column in CATEGORICAL + [TARGET]:
        rows.append({'feature': column, 'metric': 'categorical_TV',
                      'distance': total_variation(train_ref[column], synthetic_ref[column])})
    return {'mean_ks_tv': float(np.nanmean([r['distance'] for r in rows])),
        'train_exact_match_rate': exact_match_rate(synthetic_ref, train_ref),
        'test_exact_match_rate_audit_only': exact_match_rate(synthetic_ref, apply_p1(test_frame[ALL_COLUMNS])),
        'synthetic_positive_rate': float(synthetic_frame[TARGET].mean()),
        'feature_metrics': rows}

results = []
quality_rows = []
for test_site in SITES:
    train_real = data[data['hospital'] != test_site].reset_index(drop=True)
    test_real = data[data['hospital'] == test_site].reset_index(drop=True)
    real_only = train_real[ALL_COLUMNS].copy()
    for model_name in ['Logistic Regression', 'LightGBM']:
        metrics = fit_and_score(real_only, test_real, model_name, 42)
        results.append({'augmentation': 'real_only', 'synthetic_ratio': 0.0,
            'generator_seed': 0, 'model': model_name, 'test_site': test_site,
            'test_rows': len(test_real), 'train_rows': len(real_only), 'synthetic_rows': 0, **metrics})
    for seed in SEEDS:
        generators = fit_generators(train_real, seed)
        for ratio in AUGMENTATION_RATIOS:
            n_synthetic = int(round(len(train_real) * ratio))
            clean = sample_synthetic(generators, train_real, n_synthetic, seed)
            hospital_like = sample_synthetic(generators, train_real, n_synthetic, seed, hospital_like=True)
            for augmentation, synthetic in [('clean_synthetic', clean), ('hospital_like', hospital_like)]:
                audit = quality_audit(train_real, synthetic, test_real)
                quality_rows.append({'augmentation': augmentation, 'synthetic_ratio': ratio,
                    'generator_seed': seed, 'test_site': test_site, 'train_rows': len(train_real),
                    'synthetic_rows': len(synthetic), 'mean_ks_tv': audit['mean_ks_tv'],
                    'train_exact_match_rate': audit['train_exact_match_rate'],
                    'test_exact_match_rate_audit_only': audit['test_exact_match_rate_audit_only'],
                    'synthetic_positive_rate': audit['synthetic_positive_rate']})
                augmented = pd.concat([real_only, synthetic], ignore_index=True)
                for model_name in ['Logistic Regression', 'LightGBM']:
                    metrics = fit_and_score(augmented, test_real, model_name, seed)
                    results.append({'augmentation': augmentation, 'synthetic_ratio': ratio,
                        'generator_seed': seed, 'model': model_name, 'test_site': test_site,
                        'test_rows': len(test_real), 'train_rows': len(augmented),
                        'synthetic_rows': len(synthetic), **metrics})
    print('Completed test site:', test_site)

results_df = pd.DataFrame(results)
quality_df = pd.DataFrame(quality_rows)
display(results_df.head())
print('Result rows:', len(results_df), '| Quality rows:', len(quality_df))

## 4. Tổng hợp, biểu đồ và lưu kết quả

Cấu hình synthetic chỉ đáng giữ nếu cải thiện nhất quán trên các hospital thật; không chọn theo một fold đơn lẻ. Ưu tiên xem `roc_auc_worst`, `recall_worst`, `brier_mean` và tổng false negatives.

In [ ]:
summary = results_df.groupby(['augmentation', 'synthetic_ratio', 'model']).agg(
    folds=('test_site', 'nunique'), runs=('test_site', 'size'),
    seed_count=('generator_seed', 'nunique'), roc_auc_mean=('roc_auc', 'mean'),
    roc_auc_std=('roc_auc', 'std'), roc_auc_worst=('roc_auc', 'min'),
    pr_auc_mean=('pr_auc', 'mean'), recall_mean=('recall', 'mean'),
    recall_std=('recall', 'std'), recall_worst=('recall', 'min'),
    specificity_mean=('specificity', 'mean'), f1_mean=('f1', 'mean'),
    brier_mean=('brier', 'mean'),
    false_negatives_mean_per_run=('false_negatives', 'mean'),
    false_negatives_total_across_runs=('false_negatives', 'sum'),
    fit_seconds_mean=('fit_seconds', 'mean')).reset_index()

baseline = summary[summary['augmentation'] == 'real_only'].set_index('model')
delta = summary.copy()
for metric in ['roc_auc_mean', 'roc_auc_worst', 'pr_auc_mean', 'recall_mean',
                'recall_worst', 'brier_mean', 'false_negatives_mean_per_run']:
    delta[f'delta_vs_real_only_{metric}'] = delta.apply(
        lambda row: row[metric] - baseline.loc[row['model'], metric], axis=1)

display(summary.sort_values(['model', 'roc_auc_worst'], ascending=[True, False]).round(6))
display(delta.round(6))

plot_data = summary[summary['augmentation'] != 'real_only'].copy()
plt.figure(figsize=(12, 6))
sns.lineplot(data=plot_data, x='synthetic_ratio', y='roc_auc_worst', hue='model',
             style='augmentation', markers=True, dashes=False)
plt.axhline(summary.loc[summary['augmentation'].eq('real_only'), 'roc_auc_worst'].min(),
            color='black', linestyle='--', label='real_only reference')
plt.title('Worst-site ROC-AUC under synthetic augmentation')
plt.xlabel('Synthetic / real-train ratio'); plt.ylabel('Worst-site ROC-AUC')
plt.grid(alpha=0.25); plt.tight_layout(); plt.show()

paths = {
    'results': OUTPUT_DIR / 'synthetic_augmentation_loco_results.csv',
    'quality': OUTPUT_DIR / 'synthetic_quality_results.csv',
    'summary': OUTPUT_DIR / 'synthetic_augmentation_summary.csv',
    'delta': OUTPUT_DIR / 'synthetic_augmentation_delta_vs_real.csv',
}
results_df.to_csv(paths['results'], index=False)
quality_df.to_csv(paths['quality'], index=False)
summary.to_csv(paths['summary'], index=False)
delta.to_csv(paths['delta'], index=False)
plt.savefig(OUTPUT_DIR / 'augmentation_worst_site_auc.png', dpi=160, bbox_inches='tight')
config = {'dataset_rows': int(len(data)), 'sites': SITES, 'preprocessing': 'P1 sentinel-aware',
          'models': ['Logistic Regression', 'LightGBM'], 'seeds': SEEDS,
          'synthetic_ratios': AUGMENTATION_RATIOS, 'threshold': 0.5,
          'hospital_like_profile': {'feature_missing_rate': 0.15, 'ca_thal_extra_missing_rate': 0.35,
              'numeric_outlier_rate': 0.04, 'numeric_rounding_rate': 0.20,
              'categorical_invalid_rate': 0.02},
          'selection_note': 'Test hospital is real and locked; test exact match is audit only.'}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
zip_path = OUTPUT_DIR / 'uci_multicenter_synthetic_augmentation_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in list(paths.values()) + [OUTPUT_DIR / 'augmentation_worst_site_auc.png', OUTPUT_DIR / 'run_config.json']:
        archive.write(path, arcname=path.name)
print('Saved artifacts:')
for path in list(paths.values()) + [OUTPUT_DIR / 'augmentation_worst_site_auc.png', OUTPUT_DIR / 'run_config.json', zip_path]:
    print(path)